# Preprocessing — Default of Credit Card Clients

`eda.ipynb` only looks at the data. It never changes anything.

**This notebook is where the changes happen.** It reads the raw `.xls` file, makes a few specific
edits, and saves the result as a clean CSV for the training script to use.

Each change below says what it does and why, so you can always see exactly what is different
between the raw file and the processed one.

| # | change | why |
| --- | --- | --- |
| 1 | rename `default payment next month` → `DEFAULT_NEXT_MONTH` | the original name has spaces in it, which are annoying to type |
| 2 | drop `ID` | it is just a row number, and a model can find fake patterns in it |
| 3 | `EDUCATION`: turn `0`, `5`, `6` into `4` ("others") | 345 people have codes the documentation never explains, and `4` already means "others" |
| 4 | `MARRIAGE`: turn `0` into `3` ("others") | same thing, 54 people |
| 5 | leave `PAY_*`, `BILL_AMT*` and `PAY_AMT*` alone | on purpose — see section 5 |

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

RAW_PATH = Path("../data/credit_card_clients.xls")
PROCESSED_PATH = Path("../data/processed/credit_card_clients.csv")

RAW_TARGET = "default payment next month"
TARGET = "DEFAULT_NEXT_MONTH"

## 1. Load the raw file

Same load as the EDA: an Excel file, the sheet called `Data`, and the real header on row 1.

We keep `raw` around unchanged. Section 6 compares against it to prove we only changed what we
meant to change.

In [2]:
raw = pd.read_excel(RAW_PATH, sheet_name="Data", header=1)
print("raw shape:", raw.shape)
raw.head(3)

raw shape: (30000, 25)


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,0,2,2682,1725,2682,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000,0


## 2. Rename the target

The column arrives as `default payment next month` — with spaces.

Spaces are a nuisance. You cannot write `df.default payment next month`, so you are stuck typing
`df["default payment next month"]` everywhere, and some tools need extra quoting to cope. Renaming
it to `DEFAULT_NEXT_MONTH` just makes life easier.

This is the one change that used to sit in `eda.ipynb`. It is only cosmetic, but the training
script depends on the name, so it should live somewhere obvious.

In [3]:
dataset = raw.rename(columns={RAW_TARGET: TARGET})

assert TARGET in dataset.columns, "target rename failed"
assert dataset[TARGET].equals(raw[RAW_TARGET]), "rename must not touch the values"
print(f"{RAW_TARGET!r} -> {TARGET!r}")

'default payment next month' -> 'DEFAULT_NEXT_MONTH'


## 3. Drop `ID`

`ID` just counts the rows: 1, 2, 3, all the way to 30000. It tells you nothing about a person.

The problem is the model does not know that. Leave `ID` in and it might learn something like "rows
above 25000 default more often" — pure coincidence, but it will score well here and then fail on
new customers. Easier to remove it now than to remember to ignore it later.

In [4]:
dataset = dataset.drop(columns=["ID"])

print("columns after drop:", dataset.shape[1], "(23 features + 1 target)")

columns after drop: 24 (23 features + 1 target)


## 4. Tidy up the category codes that were never documented

Section 3 of the EDA found codes that the official documentation never explains:

- `EDUCATION`: `0` (14 people), `5` (280 people), `6` (51 people) — 345 in total
- `MARRIAGE`: `0` (54 people)

Both columns already have an "others" bucket — `EDUCATION = 4` and `MARRIAGE = 3` — so we move the
mystery codes in there.

Why not do something else?

- **Delete those rows?** That throws away 399 real customers for no good reason.
- **Leave them as they are?** A linear model reads these numbers as an order, so it would treat
  `6` as "more educated than" `4`. That comparison is meaningless.

This is a judgement call, not a fact. It is two lines of code, so it is easy to undo if you
disagree.

In [5]:
EDUCATION_OTHER, MARRIAGE_OTHER = 4, 3
EDUCATION_UNDOCUMENTED = [0, 5, 6]
MARRIAGE_UNDOCUMENTED = [0]

before = {
    "EDUCATION": int(dataset["EDUCATION"].isin(EDUCATION_UNDOCUMENTED).sum()),
    "MARRIAGE": int(dataset["MARRIAGE"].isin(MARRIAGE_UNDOCUMENTED).sum()),
}

dataset["EDUCATION"] = dataset["EDUCATION"].replace(
    dict.fromkeys(EDUCATION_UNDOCUMENTED, EDUCATION_OTHER)
)
dataset["MARRIAGE"] = dataset["MARRIAGE"].replace(
    dict.fromkeys(MARRIAGE_UNDOCUMENTED, MARRIAGE_OTHER)
)

print("rows folded into 'others':", before)
print()
for column in ["EDUCATION", "MARRIAGE"]:
    print(f"{column} after:")
    print(dataset[column].value_counts().sort_index().to_string(), end="\n\n")

rows folded into 'others': {'EDUCATION': 345, 'MARRIAGE': 54}

EDUCATION after:
EDUCATION
1    10585
2    14030
3     4917
4      468

MARRIAGE after:
MARRIAGE
1    13659
2    15964
3      377



## 5. What we are deliberately NOT changing

Worth writing down, so nobody later wonders whether these were just overlooked.

**The `PAY_*` codes `-2` and `0`.** These are undocumented too, and together they cover most of the
data. The usual reading is `-2` = card barely used and `0` = paid the minimum. But these columns
are the strongest predictor in the whole dataset, so merging codes here would throw away real
information. Leave them alone and let the model work it out.

**Negative `BILL_AMT` values** (roughly 600 per month). These are overpayments and refunds. Real
numbers, not mistakes.

**The 35 rows that look identical once `ID` is removed.** With 30,000 rows of mostly small whole
numbers, a few matching by chance is normal. Deleting them would mean inventing a rule the data
does not support.

**No scaling, no one-hot encoding, no train/test split.** These all belong in the training script
instead.

Here is why, using scaling as the example. Scaling squashes numbers into a similar range, so
`LIMIT_BAL` (up to 1,000,000) does not drown out `AGE` (21 to 79). To do that you need the column's
average. If you calculate that average across all 30,000 rows and only split into train and test
afterwards, your training data has quietly absorbed information about the test rows. Your test
score then looks great but is not real, because the model effectively peeked at the answers.

The fix is to split the data first, then calculate the average using the training rows only. That
can only happen in the training script, so it does not belong here.

## 6. Check we only changed what we meant to

The `assert` lines below are safety checks. Each one states something that should still be true
after our edits — the same number of rows, the target values untouched, and so on.

If someone edits this notebook later and accidentally breaks one of those rules, the cell stops
with an error right here. That is the whole point: much better to fail loudly now than to quietly
save a broken file that nobody notices until the model is already training on it.

In [6]:
UNTOUCHED = [c for c in raw.columns if c not in {"ID", RAW_TARGET, "EDUCATION", "MARRIAGE"}]

assert len(dataset) == len(raw) == 30000, "row count must not change"
assert dataset.shape[1] == 24, "expected 23 features + 1 target"
assert dataset.isna().sum().sum() == 0, "preprocessing must not introduce nulls"
assert dataset[TARGET].equals(raw[RAW_TARGET]), "target values must be unchanged"
assert dataset[UNTOUCHED].equals(raw[UNTOUCHED]), "untouched columns must be byte-identical"
assert set(dataset["EDUCATION"].unique()) <= {1, 2, 3, 4}, "EDUCATION outside documented codes"
assert set(dataset["MARRIAGE"].unique()) <= {1, 2, 3}, "MARRIAGE outside documented codes"
assert dataset["EDUCATION"].value_counts().sum() == 30000

recoded = ["EDUCATION", "MARRIAGE"]
print("all checks passed")
print("columns recoded :", recoded)
print("columns renamed :", {RAW_TARGET: TARGET})
print("columns dropped :", sorted(set(raw.columns) - set(dataset.columns) - {RAW_TARGET}))
print("rows            :", len(dataset))

all checks passed
columns recoded : ['EDUCATION', 'MARRIAGE']
columns renamed : {'default payment next month': 'DEFAULT_NEXT_MONTH'}
columns dropped : ['ID']
rows            : 30000


## 7. Save the result

CSV, because it opens in anything and needs no extra libraries. The file is about 2.7 MB.

`data/` is in `.gitignore`, so this file is not committed to git. Just re-run this notebook to
rebuild it.

The last check reads the file back and compares it to what we saved. CSV does not store data types,
so this catches anything that got mangled on the way out.

In [7]:
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
dataset.to_csv(PROCESSED_PATH, index=False)

reloaded = pd.read_csv(PROCESSED_PATH)
assert reloaded.equals(dataset), "round-trip through CSV changed the data"

print(f"wrote {PROCESSED_PATH} ({PROCESSED_PATH.stat().st_size / 1_000_000:.1f} MB)")
print("round-trip verified:", reloaded.shape)
reloaded.head(3)

wrote ../data/processed/credit_card_clients.csv (2.7 MB)
round-trip verified: (30000, 24)


,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,DEFAULT_NEXT_MONTH
0,20000,2,2,1,24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,2,2682,1725,2682,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000,0


## 8. Summary

Raw `(30000, 25)` → processed `(30000, 24)`.

- `ID` dropped
- target renamed to `DEFAULT_NEXT_MONTH`
- 399 people moved into the "others" bucket for `EDUCATION` and `MARRIAGE`

Nothing else was touched, and section 6 proves it.

**Next:** step 1 of `STEPS.md` — a training script that reads
`data/processed/credit_card_clients.csv`, splits the data (keeping the same ~22% default rate in
both halves), treats `SEX` / `EDUCATION` / `MARRIAGE` as categories rather than numbers, and logs
every run to MLflow.